# Data Acquisition (XJTU Downsampling)
**Goal:** Aggregate and downsample the raw XJTU vibration data so that 1 CSV file = 1 Bearing (where 1 row represents 1 minute of operation).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
# Define input and output paths (Modify as necessary)
RAW_DATA_PATH = r"../Raw_Data/XJTU-SY" 
OUTPUT_PATH = r"../Processed_Data/Downsampled"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Dataset Specifics
# XJTU: 25.6 kHz, ~1.28 sec snapshot (32,768 rows) every 1 minute.
# We will simply load and structure these files, potentially extracting 
# baseline representations (like saving it for feature extraction later).
ROWS_PER_FILE = 32768

print(f"Raw Data Path: {os.path.abspath(RAW_DATA_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")

In [ ]:
def process_and_combine_bearing_data():
    """
    Loops through Condition folders -> Bearing folders -> .csv files.
    Aggregates the raw 32,768 row arrays of vibration data into simpler strings
    and saves them per bearing in the corresponding Condition subfolder.
    """
    if not os.path.exists(RAW_DATA_PATH):
        print(f"Warning: Raw data path '{RAW_DATA_PATH}' does not exist. Please double-check.")
        return

    # Track results for summary
    summary_data = []
    sample_df = None

    # Use os.walk or listdir to find condition folders
    condition_folders = [f for f in os.listdir(RAW_DATA_PATH) if os.path.isdir(os.path.join(RAW_DATA_PATH, f))]
    
    for condition in condition_folders:
        cond_path = os.path.join(RAW_DATA_PATH, condition)
        
        # Create output subdirectory for this condition
        cond_output_path = os.path.join(OUTPUT_PATH, condition)
        os.makedirs(cond_output_path, exist_ok=True)
        
        bearing_folders = [f for f in os.listdir(cond_path) if os.path.isdir(os.path.join(cond_path, f))]
        
        for bearing in tqdm(bearing_folders, desc=f"Processing {condition}"):
            bearing_path = os.path.join(cond_path, bearing)
            csv_files = sorted(glob.glob(os.path.join(bearing_path, "*.csv")))
            
            if not csv_files:
                continue
                
            bearing_data_list = []
            
            for minute_idx, file in enumerate(csv_files):
                df_min = pd.read_csv(file)
                
                if 'Horizontal_vibration_signals' in df_min.columns:
                    h_sig = df_min['Horizontal_vibration_signals'].values
                    v_sig = df_min['Vertical_vibration_signals'].values
                else:
                    h_sig = df_min.iloc[:, 0].values
                    v_sig = df_min.iloc[:, 1].values if df_min.shape[1] > 1 else np.zeros_like(h_sig)
                
                minute_record = {
                    'Minute': minute_idx + 1,
                    'H_Signal': ",".join(map(str, h_sig)),
                    'V_Signal': ",".join(map(str, v_sig))
                }
                bearing_data_list.append(minute_record)
            
            if bearing_data_list:
                final_df = pd.DataFrame(bearing_data_list)
                output_filename = os.path.join(cond_output_path, f"{bearing}_downsampled.csv")
                final_df.to_csv(output_filename, index=False)
                
                file_size_mb = os.path.getsize(output_filename) / (1024 * 1024)
                summary_data.append({
                    'Condition': condition,
                    'Bearing': bearing,
                    'Rows (Minutes)': len(final_df),
                    'File Size (MB)': round(file_size_mb, 2),
                    'Path': output_filename
                })
                
                if sample_df is None:
                    sample_df = final_df

    # --- SUMMARY ---
    print("\n==========================================")
    print("PROCESSING SUMMARY")
    print("==========================================")
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
    
    print("\n==========================================")
    print("SAMPLE DATA PREVIEW (First 5 rows of output)")
    print("==========================================")
    if sample_df is not None:
        display(sample_df.head(5))
        # Optional: Print length of first H_Signal to demonstrate truncation/length
        first_h_sig = sample_df.iloc[0]['H_Signal']
        print(f"\nNote: 'H_Signal' column contains CSV string representing arrays.")

# Execute the pipeline
process_and_combine_bearing_data()